In [1]:
import glob
from general import *
from generation import *
from storage import *
from systemCopy1 import *
from copy import deepcopy

In [3]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
load_mw = Load_Data(years=2033).load_MW
#load_mw['load_MW'] = 20

In [5]:
load_mw

,load_MW
Date/time,
2033-01-01 00:30:00,61.873062
2033-01-01 01:30:00,62.962898
2033-01-01 02:30:00,62.851831
2033-01-01 03:30:00,62.807868
2033-01-01 04:30:00,64.001828
...,...
2033-12-31 19:30:00,63.699321
2033-12-31 20:30:00,63.543213
2033-12-31 21:30:00,63.384774


In [6]:
Kfiles = glob.glob('/projects/wg-ASGARD/weather_data/KAFB/*2013*')

In [7]:
KAFB_site = Site('KAFB',Weather_Data(Kfiles),-7)

In [8]:
#modify CSP tower

In [10]:
csp_config = {'thermal_power_MW_t': 100,
          'power_rating_MW_e': 50,
          'dni_des':950,
          'tower_height_m': 117,
          'receiver_width_m':5,
          'receiver_height_m':5,
          'accept_ang_y':180,
          'accept_ang_x':180,
          'heliostat_width_m':5,
          'heliostat_height_m':5,
          'field_max_scaled_rad':8,
          'layout_method':'Radial Stagger',
          'field_shape':'Hexagon',
          'interaction_limit':50,
          'row_spacing_x':1.5,
          'row_spacing_y':1.5,
          'receiver_dT': 1,
          'media_cp': 1,
          'optical_height_m': 1,}

In [11]:
ASGARD_CSP = CSP_System(name='ASGARD_CSP',
                        site = KAFB_site,
                        config = csp_config,
                        off_grid_operation = True)

In [12]:
# loads the above csp system. will need to comment this block if you change the above.
# import pickle
# with open('/projects/wg-ASGARD/csp_obj/csp_100MWhth_50MW.pkl','rb') as file:
#     ASGARD_CSP = pickle.load(file)

In [13]:
ASGARD_TES = TES_System(name='ASGARD_TES',
                        site = KAFB_site,
                        capacity_MWh_e = 570,
                        power_rating_MW_e = 50,
                        power_minimum_MW_e = 0,
                        percent_discharge_depth = 95,
                        percent_heat_loss_daily = 1,
                        charge_rate_CSP_MW_t = .6*143,
                        charge_efficiency_t2TES = .98,
                        charge_rate_resistive_MW_e = None,
                        charge_efficiency_e2TES = .98,
                        systems_charging = ['ASGARD_CSP'],
                        start_full = True,
                        off_grid_operation = True,
                        DOE_2030_targets = False,
                        dT_s = 200, # [C] dT across storage bins
                        cp_s = 1.15, # [kJ/kgK] storage media specific heat
                        rho_s = 2000) # [kg/m3] media bulk density)

In [14]:
ASGARD_systems = ['ASGARD_TES', 'ASGARD_CSP']
# ASGARD_systems = ['ASGARD_PV','ASGARD_CSP','ASGARD_TES','ASGARD_BES']

In [15]:
#need to change site for this configuration.
# ASGARD_CSP.site = KAFB_site

In [16]:
systems=ASGARD_systems
ASGARD = System(load_MW = load_mw, time_series = None,
             systems_load_order = [globals()[sys] for sys in systems])

8760


In [17]:
ts = ASGARD.timeseries

In [18]:
ts.columns

Index(['load_MW', 'target_load_MW', 'grid_to_load_MWh_e',
       'unmet_target_load_MWh_e', 'export_energy_MWh_e',
       'electricity_sale_in_hour', 'ASGARD_CSP_heat_MW_t',
       'ASGARD_CSP_heat_unused_MWh_t', 'ASGARD_CSP_to_ASGARD_TES_MWh_t',
       'ASGARD_TES_MWh_t', 'ASGARD_TES_thermal_loss_MWh_t',
       'CSP_System_to_ASGARD_TES_MWh_t', 'ASGARD_TES_to_load_MWh_t',
       'ASGARD_TES_to_load_MWh_e', 'ASGARD_TES_to_load_MWh_gas_e',
       'ASGARD_TES_to_load_MWh_gas_t', 'ASGARD_TES_to_load_MWh_gas_kg',
       'ASGARD_TES_gas_dollars', 'ASGARD_TES_frac_nogas',
       'ASGARD_TES_frac_gas', 'KAFB_POI_MW', 'unmet_load_MWh_e'],
      dtype='object')

In [19]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
ts.head(50)

,load_MW,target_load_MW,grid_to_load_MWh_e,unmet_target_load_MWh_e,export_energy_MWh_e,electricity_sale_in_hour,ASGARD_CSP_heat_MW_t,ASGARD_CSP_heat_unused_MWh_t,ASGARD_CSP_to_ASGARD_TES_MWh_t,ASGARD_TES_MWh_t,ASGARD_TES_thermal_loss_MWh_t,CSP_System_to_ASGARD_TES_MWh_t,ASGARD_TES_to_load_MWh_t,ASGARD_TES_to_load_MWh_e,ASGARD_TES_to_load_MWh_gas_e,ASGARD_TES_to_load_MWh_gas_t,ASGARD_TES_to_load_MWh_gas_kg,ASGARD_TES_gas_dollars,ASGARD_TES_frac_nogas,ASGARD_TES_frac_gas,KAFB_POI_MW,unmet_load_MWh_e
Date/time,,,,,,,,,,,,,,,,,,,,,,
2033-01-01 00:30:00,61.873062,None,0.000000,0.0,0,0.000000,0.0,0.0,0,570.000000,0.000000,0,0.000000,0.00000,0,0,0,0,0.000000,0.000000,0.00000,NaN
2033-01-01 01:30:00,62.962898,None,12.962898,NaN,0,3500.000000,0.0,0.0,0,479.278288,0.237500,0,90.484212,50.00000,0,0,0,0,1.000000,1.000000,50.00000,12.962898
2033-01-01 02:30:00,62.851831,None,12.851831,NaN,0,3500.000000,0.0,0.0,0,388.431581,0.199699,0,90.647008,50.00000,0,0,0,0,1.000000,1.000000,50.00000,12.851831
2033-01-01 03:30:00,62.807868,None,12.807868,NaN,0,3500.000000,0.0,0.0,0,297.398650,0.161846,0,90.871085,50.00000,0,0,0,0,1.000000,1.000000,50.00000,12.807868
2033-01-01 04:30:00,64.001828,None,14.001828,NaN,0,3500.000000,0.0,0.0,0,206.217739,0.123916,0,91.056994,50.00000,0,0,0,0,1.000000,1.000000,50.00000,14.001828
2033-01-01 05:30:00,64.496998,None,14.496998,NaN,0,3500.000000,0.0,0.0,0,114.849726,0.085924,0,91.282089,50.00000,0,0,0,0,1.000000,1.000000,50.00000,14.496998
2033-01-01 06:30:00,63.874565,None,16.694375,NaN,0,3302.613327,0.0,0.0,0,28.500000,0.047854,0,86.301872,47.18019,0,0,0,0,0.943604,0.943604,47.18019,16.694375
2033-01-01 07:30:00,64.300318,None,64.300318,NaN,0,0.000000,0.0,0.0,0,28.488125,0.011875,0,0.000000,0.00000,0,0,0,0,0.000000,0.000000,0.00000,64.300318
2033-01-01 08:30:00,63.425673,None,63.425673,NaN,0,0.000000,0.0,0.0,0,28.476255,0.011870,0,0.000000,0.00000,0,0,0,0,0.000000,0.000000,0.00000,63.425673
